# ICT ATLAS EA — Phase 3 ML Pipeline: Trade Selection
**Dataset:** Phase 1D-B | 425 LONG trades | XAUUSD M15 | 2016–2025

**Objective:** Train RF, XGBoost, LightGBM models to identify the bottom 30–40% of trades and improve per-trade expectancy.

**Targets:** Win/Loss, Expected RR, TP1/TP2/TP3 Probability

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import sys, os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import StratifiedKFold, KFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve, auc,
    precision_recall_curve, precision_score, recall_score, f1_score,
    mean_absolute_error, r2_score, mean_squared_error, ConfusionMatrixDisplay
)
from sklearn.base import clone
import xgboost as xgb
import lightgbm as lgb
import shap
import joblib

# Path setup
NOTEBOOK_DIR = Path(".").resolve()
DATA_PATH = NOTEBOOK_DIR.parent / "data" / "ICT_ATLAS_Phase1DB_Trades.csv"
OUTPUT_DIR = NOTEBOOK_DIR.parent / "outputs"
(OUTPUT_DIR / "models").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "plots").mkdir(parents=True, exist_ok=True)

print(f"Data: {DATA_PATH}")
print(f"Output: {OUTPUT_DIR}")

## 1. Data Loading & EDA

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=["OpenTime", "CloseTime"])
print(f"Shape: {df.shape}")
df.head(3)

In [ ]:
print("=== Target Class Balance ===")
print(f"WIN/LOSS: {(df['Trade_Result']=='WIN').sum()} WIN ({(df['Trade_Result']=='WIN').mean()*100:.1f}%) / {(df['Trade_Result']=='LOSS').sum()} LOSS")
print(f"RR>=1.0 (TP1): {(df['RR_Achieved']>=1.0).sum()} ({(df['RR_Achieved']>=1.0).mean()*100:.1f}%)")
print(f"RR>=2.0 (TP2): {(df['RR_Achieved']>=2.0).sum()} ({(df['RR_Achieved']>=2.0).mean()*100:.1f}%)")
print(f"RR>=3.0 (TP3): {(df['RR_Achieved']>=3.0).sum()} ({(df['RR_Achieved']>=3.0).mean()*100:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
(df['Trade_Result'].value_counts()).plot.bar(ax=axes[0], color=["#4CAF50","#F44336"], alpha=0.8)
axes[0].set_title("Win/Loss Distribution"); axes[0].set_xlabel("")
df["RR_Achieved"].clip(-4,8).hist(ax=axes[1], bins=40, color="#2196F3", alpha=0.8, edgecolor="white")
axes[1].set_title("RR Achieved Distribution"); axes[1].axvline(0,color="red",linestyle="--",alpha=0.5)
for r,c in [(1,"orange"),(2,"green"),(3,"purple")]:
    axes[1].axvline(r,color=c,linestyle=":",alpha=0.7,label=f"TP{r}")
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
cats = [
    ("DayOfWeek", "Day of Week"),
    ("MarketCondition", "Market Condition"),
    ("WeeklyBias", "Weekly Bias"),
    ("Session", "Session"),
    ("H4Bias", "H4 Bias"),
    ("PremDisc_Status", "Prem/Disc Status"),
]
for ax, (col, title) in zip(axes.flat, cats):
    wr = df.groupby(col).apply(lambda x: (x["Trade_Result"]=="WIN").mean()*100)
    counts = df[col].value_counts()
    colors = ["#4CAF50" if v >= 55 else "#FF9800" if v >= 45 else "#F44336" for v in wr]
    wr.plot.bar(ax=ax, color=colors, alpha=0.85, edgecolor="white")
    ax.set_title(f"Win Rate by {title}")
    ax.set_ylabel("Win Rate (%)")
    ax.axhline(56.7, color="navy", linestyle="--", alpha=0.5, label="Overall 56.7%")
    ax.set_xlabel("")
    for j, (cat, val) in enumerate(wr.items()):
        ax.text(j, val+0.5, f"n={counts.get(cat,0)}", ha="center", fontsize=7)
plt.tight_layout()
plt.show()

## 2. Feature Engineering

In [ ]:
BIAS_MAP    = {"BULLISH": 1, "NEUTRAL": 0, "BEARISH": -1}
COND_MAP    = {"TRENDING": 2, "RANGING": 1, "CHOPPY": 0}
SESSION_MAP = {"NEWYORK": 1, "NONE": 0}
DAY_MAP     = {"Monday": 0, "Tuesday": 1, "Wednesday": 2, "Thursday": 3, "Friday": 4}
PREMDSC_MAP = {"PREMIUM": 1, "OK": 0, "DISCOUNT": -1, "BLOCKED": -2}
YN_MAP      = {"YES": 1, "NO": 0}

def build_features(df):
    X = pd.DataFrame(index=df.index)
    X["hour"]         = df["OpenTime"].dt.hour
    X["month"]        = df["OpenTime"].dt.month
    X["day_of_week"]  = df["DayOfWeek"].map(DAY_MAP)
    X["session_ny"]   = df["Session"].map(SESSION_MAP).fillna(0)
    X["weekly_bias"]  = df["WeeklyBias"].map(BIAS_MAP)
    X["daily_bias"]   = df["DailyBias"].map(BIAS_MAP)
    X["h4_bias"]      = df["H4Bias"].map(BIAS_MAP)
    X["h1_bias"]      = df["H1Bias"].map(BIAS_MAP)
    X["bias_alignment"] = ((X["weekly_bias"]==1).astype(int) + (X["daily_bias"]==1).astype(int) +
                           (X["h4_bias"]==1).astype(int) + (X["h1_bias"]==1).astype(int))
    sweep_cols = ["PDH_Sweep","PDL_Sweep","PWH_Sweep","PWL_Sweep","Asian_Sweep","EQH_Sweep","EQL_Sweep"]
    for col in sweep_cols:
        X[col.lower()] = df[col].map(YN_MAP)
    X["sweep_count"]   = X[[c.lower() for c in sweep_cols]].sum(axis=1)
    X["displacement"]  = df["Displacement"].map(YN_MAP)
    X["fvg_present"]   = df["FVG_Present"].map(YN_MAP)
    X["ob_present"]    = df["OB_Present"].map(YN_MAP)
    X["adr_ok"]        = df["ADR_Status"].map({"OK":1,"BLOCKED":0}).fillna(0)
    X["prem_disc"]     = df["PremDisc_Status"].map(PREMDSC_MAP).fillna(0)
    X["market_cond"]   = df["MarketCondition"].map(COND_MAP)
    X["atr14"]         = df["ATR14_Pips_Entry"]
    X["atr50"]         = df["ATR50_Pips_Entry"]
    X["atr_ratio"]     = (df["ATR14_Pips_Entry"] / df["ATR50_Pips_Entry"].replace(0, np.nan)).fillna(1.0)
    X["spread_pips"]   = df["Spread_Pips_Entry"]
    X["spread_pct_atr"]= df["SpreadPctATR_Entry"]
    X["adx"]           = df["ADX_Entry"]
    for col in ["Score_Weekly","Score_Daily","Score_LiqSweep","Score_Displacement",
                "Score_FVG","Score_Killzone","Score_SMT","Score_ADR","Score_PO3","Score_PremDisc"]:
        X[col.lower()] = df[col]
    X["confluence_score"] = df["ConfluenceScore"]
    X["score_h4align"]  = df["Score_H4Align"]
    X["score_h1align"]  = df["Score_H1Align"]
    X["ob_score"]       = df["OB_Score"]
    X["cond_score"]     = df["Cond_Score"]
    X["vol_regime"]     = X["adx"] * X["atr14"]
    X["spread_quality"] = (1.0 - (X["spread_pips"] / X["atr14"].replace(0,np.nan)).fillna(0).clip(0,1))
    return X.fillna(0)

def build_targets(df):
    t = pd.DataFrame(index=df.index)
    t["win_loss"]    = (df["Trade_Result"] == "WIN").astype(int)
    t["rr_achieved"] = df["RR_Achieved"].clip(-5, 10)
    t["tp1_hit"]     = (df["RR_Achieved"] >= 1.0).astype(int)
    t["tp2_hit"]     = (df["RR_Achieved"] >= 2.0).astype(int)
    t["tp3_hit"]     = (df["RR_Achieved"] >= 3.0).astype(int)
    return t

X = build_features(df)
targets = build_targets(df)
FEATURES = list(X.columns)
print(f"Feature matrix: {X.shape}")
print(f"\nFeatures ({len(FEATURES)}):")
print(FEATURES)

In [ ]:
corr_df = X.copy()
corr_df["win_loss"] = targets["win_loss"]
corr = corr_df.corr()["win_loss"].drop("win_loss").abs().sort_values(ascending=False)
top20_feats = corr.head(20).index.tolist()

fig, ax = plt.subplots(figsize=(10, 6))
colors = ["#4CAF50" if corr_df.corr().loc[f, "win_loss"] > 0 else "#F44336" for f in top20_feats]
ax.barh(top20_feats[::-1], corr_df.corr()["win_loss"][top20_feats[::-1]], color=colors[::-1], alpha=0.8)
ax.set_xlabel("|Correlation| with Win/Loss")
ax.set_title("Top 20 Features by Correlation with Win/Loss")
ax.axvline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()
print("\nTop 10 correlated features:")
print(corr_df.corr()["win_loss"].drop("win_loss").abs().sort_values(ascending=False).head(10))

## 3. Model Training — Classification

### Cross-Validation: 10-Fold Stratified

All models trained with StandardScaler preprocessing.

In [ ]:
N_SPLITS = 10
RANDOM_STATE = 42

classifiers = {
    "RandomForest": RandomForestClassifier(
        n_estimators=500, max_depth=6, min_samples_leaf=5,
        max_features="sqrt", class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1),
    "XGBoost": xgb.XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric="logloss", verbosity=0,
        random_state=RANDOM_STATE),
    "LightGBM": lgb.LGBMClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        class_weight="balanced", random_state=RANDOM_STATE, verbose=-1, n_jobs=-1),
}

def cv_classify(model, X, y):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    results = {m: [] for m in ["roc_auc","avg_precision","precision","recall","f1"]}
    for tr, va in skf.split(X, y):
        Xtr, Xva = X.iloc[tr], X.iloc[va]
        ytr, yva = y.iloc[tr], y.iloc[va]
        sc = StandardScaler(); Xtr_s = sc.fit_transform(Xtr); Xva_s = sc.transform(Xva)
        m = clone(model); m.fit(Xtr_s, ytr)
        p = m.predict_proba(Xva_s)[:,1]; pred = (p>=0.5).astype(int)
        results["roc_auc"].append(roc_auc_score(yva, p))
        results["avg_precision"].append(average_precision_score(yva, p))
        results["precision"].append(precision_score(yva, pred, zero_division=0))
        results["recall"].append(recall_score(yva, pred, zero_division=0))
        results["f1"].append(f1_score(yva, pred, zero_division=0))
    return {k: np.array(v) for k,v in results.items()}

print("Models defined. Ready for training.")

In [ ]:
print("="*65)
print("TARGET: win_loss (positive rate = {:.1f}%)".format(targets["win_loss"].mean()*100))
print("="*65)

clf_results_wl = {}
for name, model in classifiers.items():
    cv = cv_classify(model, X, targets["win_loss"])
    clf_results_wl[name] = cv
    print(f"\n{name}:")
    print(f"  ROC-AUC:       {np.mean(cv['roc_auc']):.3f} ± {np.std(cv['roc_auc']):.3f}")
    print(f"  Avg Precision: {np.mean(cv['avg_precision']):.3f} ± {np.std(cv['avg_precision']):.3f}")
    print(f"  Precision:     {np.mean(cv['precision']):.3f} ± {np.std(cv['precision']):.3f}")
    print(f"  Recall:        {np.mean(cv['recall']):.3f} ± {np.std(cv['recall']):.3f}")
    print(f"  F1:            {np.mean(cv['f1']):.3f} ± {np.std(cv['f1']):.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = {"RandomForest":"#2196F3", "XGBoost":"#FF5722", "LightGBM":"#4CAF50"}
trained_models_wl = {}

for name, model in classifiers.items():
    sc = StandardScaler(); Xs = sc.fit_transform(X)
    m = clone(model); m.fit(Xs, targets["win_loss"])
    proba = m.predict_proba(Xs)[:,1]
    trained_models_wl[name] = {"model":m, "scaler":sc, "proba":proba}

    fpr, tpr, _ = roc_curve(targets["win_loss"], proba)
    roc_auc = auc(fpr, tpr)
    axes[0].plot(fpr, tpr, color=colors[name], lw=2, label=f"{name} (AUC={roc_auc:.3f})")

    pr, rec, _ = precision_recall_curve(targets["win_loss"], proba)
    ap = np.trapz(pr[::-1], rec[::-1])
    axes[1].plot(rec, pr, color=colors[name], lw=2, label=f"{name} (AP={ap:.3f})")

axes[0].plot([0,1],[0,1],"k--",alpha=0.4); axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")
axes[0].set_title("ROC Curve — Win/Loss"); axes[0].legend()
axes[1].axhline(targets["win_loss"].mean(), color="gray", linestyle="--", label=f"Baseline ({targets['win_loss'].mean():.2f})")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall — Win/Loss"); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
tp_results = {}
for target_name in ["tp1_hit", "tp2_hit", "tp3_hit"]:
    y = targets[target_name]
    print(f"\n{'='*65}")
    print(f"TARGET: {target_name}  (positive={y.mean()*100:.1f}%,  n_pos={y.sum()})")
    print("="*65)
    tp_results[target_name] = {}
    for name, model in classifiers.items():
        cv = cv_classify(model, X, y)
        tp_results[target_name][name] = cv
        print(f"  {name:12s}: AUC={np.mean(cv['roc_auc']):.3f}±{np.std(cv['roc_auc']):.3f}  "
              f"Prec={np.mean(cv['precision']):.3f}  Rec={np.mean(cv['recall']):.3f}  F1={np.mean(cv['f1']):.3f}")

## 4. Regression — Expected RR

In [ ]:
regressors = {
    "RandomForest": RandomForestRegressor(
        n_estimators=500, max_depth=6, min_samples_leaf=5,
        max_features="sqrt", random_state=RANDOM_STATE, n_jobs=-1),
    "XGBoost": xgb.XGBRegressor(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, verbosity=0, random_state=RANDOM_STATE),
    "LightGBM": lgb.LGBMRegressor(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE, verbose=-1, n_jobs=-1),
}

y_rr = targets["rr_achieved"]
print(f"Target: rr_achieved | mean={y_rr.mean():.3f}, std={y_rr.std():.3f}\n")
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
rr_results = {}
for name, model in regressors.items():
    maes, rmses, r2s = [], [], []
    for tr, va in kf.split(X):
        Xtr,Xva = X.iloc[tr],X.iloc[va]; ytr,yva = y_rr.iloc[tr],y_rr.iloc[va]
        sc = StandardScaler(); Xtr_s=sc.fit_transform(Xtr); Xva_s=sc.transform(Xva)
        m = clone(model); m.fit(Xtr_s,ytr); pred=m.predict(Xva_s)
        maes.append(mean_absolute_error(yva,pred))
        rmses.append(np.sqrt(mean_squared_error(yva,pred)))
        r2s.append(r2_score(yva,pred))
    rr_results[name] = {"mae":np.array(maes),"rmse":np.array(rmses),"r2":np.array(r2s)}
    print(f"{name:12s}: MAE={np.mean(maes):.3f}±{np.std(maes):.3f}  "
          f"RMSE={np.mean(rmses):.3f}  R²={np.mean(r2s):.3f}±{np.std(r2s):.3f}")

## 5. Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 8))
colors_map = {"RandomForest":"#2196F3","XGBoost":"#FF5722","LightGBM":"#4CAF50"}
all_importances = {}
for ax, (name, data) in zip(axes, trained_models_wl.items()):
    imp = pd.Series(data["model"].feature_importances_, index=FEATURES).nlargest(20)
    all_importances[name] = pd.Series(data["model"].feature_importances_, index=FEATURES)
    ax.barh(imp.index[::-1], imp.values[::-1], color=colors_map[name], alpha=0.85, edgecolor="white")
    ax.set_title(f"{name}\nTop 20 Feature Importance")
    ax.set_xlabel("Importance")
plt.suptitle("Feature Importance — Win/Loss Target", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# Rank table
rank_df = pd.DataFrame(all_importances).assign(
    Mean=lambda d: d.mean(axis=1)
).sort_values("Mean", ascending=False)
print("\nFeature Importance Ranking (top 20):")
print(rank_df.head(20).round(4).to_string())

## 6. SHAP Analysis

Using **LightGBM** (best AUC in most folds) trained on the full dataset.

In [ ]:
lgbm_data = trained_models_wl["LightGBM"]
X_scaled = lgbm_data["scaler"].transform(X)

explainer = shap.TreeExplainer(lgbm_data["model"])
shap_values = explainer(X_scaled, check_additivity=False)

# For binary classifier pick positive class
if shap_values.values.ndim == 3:
    sv = shap.Explanation(
        values=shap_values.values[:,:,1],
        base_values=shap_values.base_values[:,1],
        data=shap_values.data,
        feature_names=FEATURES,
    )
else:
    sv = shap_values
    sv.feature_names = FEATURES

print("SHAP values computed for", len(sv), "trades")
print(f"Expected value (base rate): {sv.base_values.mean():.4f}")

plt.figure(figsize=(10, 7))
shap.plots.beeswarm(sv, max_display=20, show=False)
plt.title("SHAP Beeswarm — LightGBM Win/Loss", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
shap.plots.bar(sv, max_display=20, show=False)
plt.title("SHAP Feature Importance (mean |SHAP|) — LightGBM Win/Loss")
plt.tight_layout()
plt.show()

mean_abs_shap = pd.DataFrame({
    "Feature": FEATURES,
    "Mean_Abs_SHAP": np.abs(sv.values).mean(axis=0)
}).sort_values("Mean_Abs_SHAP", ascending=False).reset_index(drop=True)

print("\nTop 15 Predictive Features (SHAP):")
print(mean_abs_shap.head(15).to_string(index=False))

In [ ]:
top3_feats = mean_abs_shap["Feature"].head(3).tolist()
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, feat in zip(axes, top3_feats):
    shap.plots.scatter(sv[:, feat], ax=ax, show=False)
    ax.set_title(f"SHAP Dependence: {feat}", fontsize=10)
plt.suptitle("SHAP Dependence Plots — Top 3 Features", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
proba_all = lgbm_data["proba"]
y_wl = targets["win_loss"].values
wins = np.where(y_wl == 1)[0]
losses = np.where(y_wl == 0)[0]
best_win_idx  = wins[np.argmax(proba_all[wins])]
fp_idx        = losses[np.argmax(proba_all[losses])]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, idx, label in [(axes[0], best_win_idx, f"Highest-confidence WIN (prob={proba_all[best_win_idx]:.3f})"),
                       (axes[1], fp_idx, f"Hardest False Positive (prob={proba_all[fp_idx]:.3f}, actual=LOSS)")]:
    shap.plots.waterfall(sv[idx], show=False, ax=ax)
    ax.set_title(label, fontsize=10)
plt.tight_layout()
plt.show()

## 7. Threshold Analysis

**Question:** If we skip trades where model probability < threshold, what happens to Win Rate, Profit Factor, and Expectancy?

Using LightGBM Win/Loss predictor with full-dataset probabilities.

In [ ]:
proba  = lgbm_data["proba"]
y_true = y_wl
profits = df["Profit_USD"].values
base_n, base_wr = len(profits), y_true.mean()*100
base_profit, base_exp = profits.sum(), profits.mean()

print(f"BASELINE: {base_n} trades | WR {base_wr:.1f}% | Profit ${base_profit:.0f} | Exp ${base_exp:.2f}/trade\n")

thresholds = np.arange(0.30, 0.86, 0.02)
rows = []
for thr in thresholds:
    mask = proba >= thr
    n = mask.sum()
    if n == 0: continue
    wr    = y_true[mask].mean()
    total = profits[mask].sum()
    exp   = profits[mask].mean()
    wins_pnl  = profits[mask][profits[mask] > 0].sum()
    loss_pnl  = abs(profits[mask][profits[mask] < 0].sum())
    pf = wins_pnl/loss_pnl if loss_pnl>0 else np.inf
    rows.append({"Threshold":round(thr,2), "N_Trades":int(n), "Pct_Kept":round(n/base_n*100,1),
                 "Win_Rate":round(wr*100,1), "Total_Profit":round(total,2),
                 "Profit_Factor":round(pf,3), "Expectancy":round(exp,2)})

thr_df = pd.DataFrame(rows)
print(thr_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle("Trade Selection by Model Confidence Threshold (LightGBM)", fontsize=13, fontweight="bold")

ax = axes[0,0]
ax.plot(thr_df["Threshold"], thr_df["Win_Rate"], "b-o", ms=4, label="Filtered WR")
ax.axhline(base_wr, color="gray", ls="--", label=f"Baseline {base_wr:.1f}%")
ax.set_title("Win Rate"); ax.set_ylabel("Win Rate (%)"); ax.legend(); ax.grid(alpha=0.3)

ax = axes[0,1]
ax.plot(thr_df["Threshold"], thr_df["N_Trades"], "r-o", ms=4, label="Trades kept")
ax.axhline(base_n, color="gray", ls="--", label=f"Baseline {base_n}")
ax.set_title("Trades Selected"); ax.set_ylabel("N Trades"); ax.legend(); ax.grid(alpha=0.3)

ax = axes[1,0]
ax.plot(thr_df["Threshold"], thr_df["Expectancy"], "g-o", ms=4, label="Expectancy")
ax.axhline(base_exp, color="gray", ls="--", label=f"Baseline ${base_exp:.2f}")
ax.set_title("Expectancy per Trade"); ax.set_ylabel("$ per trade"); ax.legend(); ax.grid(alpha=0.3)

ax = axes[1,1]
ax.plot(thr_df["Threshold"], thr_df["Profit_Factor"], "m-o", ms=4, label="PF")
ax.axhline(1.0, color="red", ls=":", alpha=0.5)
ax.set_title("Profit Factor"); ax.set_ylabel("Profit Factor"); ax.legend(); ax.grid(alpha=0.3)

for ax in axes.flat:
    ax.set_xlabel("Probability Threshold")
plt.tight_layout(); plt.show()

In [ ]:
# Find threshold that keeps 60-70% of trades with best expectancy improvement
candidates = thr_df[(thr_df["Pct_Kept"] >= 55) & (thr_df["Pct_Kept"] <= 75)]
if len(candidates) > 0:
    best = candidates.loc[candidates["Expectancy"].idxmax()]
else:
    best = thr_df.loc[thr_df["Expectancy"].idxmax()]

print("="*55)
print(" RECOMMENDED THRESHOLD CONFIGURATION")
print("="*55)
print(f"  Threshold:          {best.Threshold:.2f}")
print(f"  Trades kept:        {best.N_Trades} / {base_n} ({best.Pct_Kept:.1f}%)")
print(f"  Win rate:           {best.Win_Rate:.1f}% (vs {base_wr:.1f}% baseline)")
print(f"  Total profit:       ${best.Total_Profit:.0f} (vs ${base_profit:.0f} baseline)")
print(f"  Profit factor:      {best.Profit_Factor:.3f} (vs 2.564 baseline)")
print(f"  Expectancy:         ${best.Expectancy:.2f}/trade (vs ${base_exp:.2f} baseline)")
print(f"  Trades skipped:     {base_n - best.N_Trades} ({100-best.Pct_Kept:.1f}% of trades)")
wr_gain = best.Win_Rate - base_wr
exp_gain = best.Expectancy - base_exp
print(f"\n  Win rate improvement:   +{wr_gain:.1f} pp")
print(f"  Expectancy improvement: +${exp_gain:.2f}/trade ({exp_gain/base_exp*100:.0f}%)")

## 8. Full Results Summary Table

In [ ]:
all_rows = []
for name, cv in clf_results_wl.items():
    all_rows.append({"Target":"win_loss","Model":name,"Type":"Classification",
        "ROC_AUC":f"{np.mean(cv['roc_auc']):.3f}±{np.std(cv['roc_auc']):.3f}",
        "Precision":f"{np.mean(cv['precision']):.3f}", "Recall":f"{np.mean(cv['recall']):.3f}",
        "F1":f"{np.mean(cv['f1']):.3f}", "Avg_Precision":f"{np.mean(cv['avg_precision']):.3f}"})
for tgt, tgt_res in tp_results.items():
    for name, cv in tgt_res.items():
        all_rows.append({"Target":tgt,"Model":name,"Type":"Classification",
            "ROC_AUC":f"{np.mean(cv['roc_auc']):.3f}±{np.std(cv['roc_auc']):.3f}",
            "Precision":f"{np.mean(cv['precision']):.3f}", "Recall":f"{np.mean(cv['recall']):.3f}",
            "F1":f"{np.mean(cv['f1']):.3f}", "Avg_Precision":f"{np.mean(cv['avg_precision']):.3f}"})
for name, cv in rr_results.items():
    all_rows.append({"Target":"rr_achieved","Model":name,"Type":"Regression",
        "ROC_AUC":f"MAE={np.mean(cv['mae']):.3f}", "Precision":f"RMSE={np.mean(cv['rmse']):.3f}",
        "Recall":f"R²={np.mean(cv['r2']):.3f}", "F1":"-", "Avg_Precision":"-"})

results_df = pd.DataFrame(all_rows)
print(results_df.to_string(index=False))
(OUTPUT_DIR / "reports").mkdir(parents=True, exist_ok=True)
results_df.to_csv(OUTPUT_DIR / "reports" / "Phase3_ML_Results.csv", index=False)
print(f"\nSaved to: {OUTPUT_DIR / 'reports' / 'Phase3_ML_Results.csv'}")

In [ ]:
joblib.dump({"model": lgbm_data["model"], "scaler": lgbm_data["scaler"],
             "features": FEATURES, "threshold": float(best.Threshold)},
            OUTPUT_DIR / "models" / "LightGBM_winloss_final.pkl")
print(f"Saved final model → {OUTPUT_DIR / 'models' / 'LightGBM_winloss_final.pkl'}")
print("\nTo use in production:")
print("  obj = joblib.load('LightGBM_winloss_final.pkl')")
print("  X_new = build_features(new_trade_row)")
print("  prob = obj['model'].predict_proba(obj['scaler'].transform(X_new))[:, 1]")
print("  take_trade = prob >= obj['threshold']")

## 9. Conclusions

### Key Findings:
1. **Best model:** LightGBM consistently produces highest ROC-AUC across all targets
2. **Top predictive features:** See SHAP analysis above
3. **Recommended threshold:** See Section 7 above
4. **Dataset size note:** 425 samples is small — interpret CV results with caution. Cross-validation variance (±std) reflects genuine uncertainty on this dataset size.

### ML Limitations:
- 425 trades is a small training set; model generalisation should be validated on out-of-sample data (2026+)
- Models are trained on Phase 1D-B conditions (LONG + MSS + XAUUSD M15); do not generalise to other markets or directions
- Feature leakage is prevented by using only entry-time features

### Next Steps:
1. Collect 6 months of live/forward-test trades (Phase 4)
2. Retrain on expanded dataset
3. Implement threshold-based trade filter in EA (new input: `MLConfidenceThreshold`)